# Lab: Cortex Agents

**This is an OPTIONAL lab. We will NOT dedicate class time for this lab. We recommend you try this lab outside of the dedicated class time. If you happen to have free extra time leftover during the class, please try this out as you wish!**

📚 In this lab you will learn and practice the following:

**Cortex Agents REST API Operations**:

- ❄️ Create a Cortex Search service for document retrieval
- ❄️ Set up REST API authentication using Programmatic Access Tokens (PAT)
- ❄️ Create Cortex Agents using the REST API
- ❄️ Configure agents with Cortex Search tools for RAG capabilities
- ❄️ Manage agent lifecycle (create, list, describe, update)
- ❄️ Test agent interactions and multi-turn conversations

**Key Concepts**:
- **Cortex Agent**: An AI assistant that can use tools (like Cortex Search) to answer questions
- **Cortex Search Service**: A vector search service that enables RAG (Retrieval Augmented Generation)
- **REST API**: The programmatic interface for managing Cortex Agents
- **Conversation Threads**: Persistent contexts that maintain conversation history

📌 **Note**:

Due to **regional workload spikes**, there may be **latency** with some of the steps. In the real world, for consistent performance, customers can explore [**Provisioned Throughput**](https://docs.snowflake.com/en/user-guide/snowflake-cortex/provisioned-throughput).

If you find your queries are running for more than 5 minutes, cancel and come back and try them later.

If you find models that are deprecated, use CoCo to help you fix the issue by selecting a suitable model.

---

### 🤖 Use CoCo as you go!

> **💡 TIP 1**: Use CoCo to explain complex SQL/Python statements. Select any code  and ask *"Explain this SQL/Python"* to get a plain-language breakdown of what it does.
>
> **💡 TIP 2**: Want to learn more about any feature? Ask CoCo *"What does [feature name] do?"* to get more details and examples.
>
> **💡 TIP 3**: If you encounter a deprecated model error, ask CoCo *"Replace deprecated models in this notebook with current similar low-cost alternatives"* and it will fix them for you.


---

## Connect to a Service

Before running cells in this notebook, you must connect to a compute service.

**First time (create a new service):**
1. Click the **Connect** button at the top of this notebook
2. Click **Create Service** — a default name like `{{user}}_SERVICE1` will be suggested
3. Click **Service Settings** and select `ALLOW_ALL_EAI` as the external access integration
4. Leave other settings as default and click **Create**
5. Wait for the service to reach a **READY** state

**Returning (service already exists):**
1. Click the **Connect** button
2. Select your existing service from the list

Once connected, you can run Python and SQL cells interactively.

## Setting Up Cortex Search Service

We will create a Cortex Search service using the PDF documents stored in our stage named **search_docs**. 

### Load the documents that are going to be used by the search service.

First, let's make sure we have some documents for the search service to run on. In our case, the Snowflake Education team has already loaded some PDF documents that contain details about the tour, and we cloned that stage in Lab 1. The documents can be found in the **{{user}}_GENAI_DB.RESOURCES.GENAI2DAY/SEARCH_DOCS**. 

Run the following commands to confirm you can see four PDF documents.

## Introduction to Cortex Agents

Cortex Agents are AI-powered conversational assistants that can interact with your Snowflake data and services. They provide a natural language interface to your data ecosystem, allowing users to ask questions and get insights without writing SQL or understanding complex data structures.

Key capabilities of Cortex Agents include:

- **Natural Language Processing**: Understand and respond to user queries in natural language
- **Tool Integration**: Connect to various Snowflake services like Cortex Search, Cortex Analyst, and custom tools
- **Conversational Memory**: Maintain context across multiple interactions
- **Flexible Configuration**: Customize behavior, instructions, and capabilities
- **REST API Management**: Programmatically create, configure, and manage agents

### Set up your current context for the role, database, schema and warehouse.

In [ ]:
from snowflake.snowpark.context import get_active_session
import requests
import json
import os

session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_genai_db'

print('Your current CONTEXT information:')
print(session)

In [ ]:
%%sql -r Set_up_your_current_context_for_the_sql
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA resources;
USE WAREHOUSE {{user}}_genai_wh;
ALTER SESSION SET query_tag = '{{user}} lab - TOPIC: Cortex Agents REST API';
SHOW PARAMETERS LIKE 'query_tag' in session 
  ->> SELECT "value" AS query_tag FROM $1;

In [ ]:
%%sql -r Setting_Up_Cortex_Search_Service_sql
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA resources;

LIST @genai2day/search_docs;

### Create a table to store parsed documents.

Next, we will create a new table called **PARSE_DOC** to parse the documents using **AI_PARSE_DOCUMENT** function. This table will include the relative path of each document, source_url, raw_text extracted from each PDF document.

In [ ]:
%%sql -r Create_a_table_to_store_the_chunked_sql
-- Create and populate PARSE_DOC table
CREATE OR REPLACE TABLE {{user}}_genai_db.raw.PARSE_DOC AS
SELECT
    relative_path AS relative_path,
    GET_PRESIGNED_URL('@{{user}}_genai_db.resources.genai2day', relative_path) AS source_url,
    size AS size,
    TO_VARCHAR(
        AI_PARSE_DOCUMENT(
            TO_FILE('@{{user}}_genai_db.resources.genai2day', relative_path),
            {'mode': 'LAYOUT'}
        ):content
    ) AS raw_text
FROM DIRECTORY(@{{user}}_genai_db.resources.genai2day)
WHERE STARTSWITH(relative_path, 'search_docs/');

### Query the contents of the PARSE_DOC table.

In [ ]:
%%sql -r Query_the_contents_of_the_PARSE_DOC_sql
SELECT *, LENGTH(raw_text) AS raw_text_size
FROM {{user}}_genai_db.raw.PARSE_DOC ;

### Create a table PARSE_DOC_CHUNKS to hold the chunked documents.

The **SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER** function is used to split the raw text into smaller chunks based on the following parameters:

The input text is treated as markdown, split into 1800-character chunks (~512 tokens) with a 300-character overlap to maintain context across chunks.

Using Snowflake's **AI_CLASSIFY**, it classifies chunks of text into one of five categories (sunsettour, mountainhike, winetour, balloonride, snorkeling) by passing the text through a machine learning model.

In [ ]:
%%sql -r Create_a_table_PARSE_DOC_CHUNKS_to_hold_sql
-- Create table with explicit column list
CREATE OR REPLACE TABLE {{user}}_genai_db.raw.parse_doc_chunks (
    doc_id,
    relative_path,
    category,
    chunk,
    source_url
) AS
SELECT
    UUID_STRING() AS doc_id,
    relative_path,
    AI_CLASSIFY(
        TO_VARCHAR(c.value),
        ['sunsettour', 'mountainhike', 'winetour', 'balloonride', 'snorkeling']
    ):labels[0]::VARCHAR AS category,
    TO_VARCHAR(c.value) AS chunk,
    source_url
FROM {{user}}_genai_db.raw.parse_doc,
     LATERAL FLATTEN(
         input => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
             raw_text,
             'markdown',
             1800,
             300
         )
     ) c;

### Query the contents of the PARSE_DOC_CHUNKS table.

In [ ]:
%%sql -r Query_the_contents_of_PARSE_DOC_CHUNKS_sql
SELECT *, LENGTH(chunk) AS chunk_size
FROM {{user}}_genai_db.raw.PARSE_DOC_CHUNKS
ORDER BY relative_path;

### Create a Cortex Search Service that runs on the table with chunked PDFs.

Next, we need to build a search service using the **CREATE SEARCH SERVICE** command. This service will generate embeddings based on the **chunk** column and retrieve information based on similarity, while using the "category" column as a filter. The search service will refresh every 30 minutes. The retrieved data will include the following:

- Doc ID
- Relative path
- Chunk 
- File URL
- Category

The search service is created with a straightforward SQL statement and automatically handles complex search index maintenance incrementally, eliminating the need for us to manage any infrastructure.

In [ ]:
%%sql -r Create_a_Cortex_Search_Service_that_sql
-- Create a Cortex Search Service
CREATE OR REPLACE CORTEX SEARCH SERVICE {{user}}_genai_db.resources.travelbug_search
ON chunk
ATTRIBUTES category
WAREHOUSE = {{user}}_genai_wh
TARGET_LAG = '30 minutes'
EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
    SELECT
        doc_id,
        chunk,
        relative_path,
        source_url,
        category::VARCHAR AS category
    FROM {{user}}_genai_db.raw.parse_doc_chunks
);

### Verify the Cortex Search Service was created successfully.

In [ ]:
%%sql -r Verify_the_Cortex_Search_Service_was_sql
-- Verify that our search service exists
SHOW CORTEX SEARCH SERVICES IN SCHEMA {{user}}_genai_db.resources;

## Setting Up REST API Authentication

Cortex Agents are managed through the Snowflake REST API. To authenticate, we use a **Programmatic Access Token (PAT)** which provides JWT-based authentication.

**Authentication Methods**:
- **PAT (Programmatic Access Token)**: Easy to generate via SQL
- **Key-pair JWT**: More secure for production

**Steps**:
1. Generate a PAT token for the current user
2. Use the token in REST API requests with Bearer authentication

### Generate Programmatic Access Token (PAT)

A PAT provides JWT-based authentication for REST API calls. The token is generated for the current user.

**Note**: PAT tokens expire after the specified `DAYS_TO_EXPIRY` (default: 15 days, max: 90 days).

In [ ]:
# Generate a Programmatic Access Token (PAT) for REST API authentication
pat_name = 'cortex_agents_lab_pat'
days_to_expiry = 1  # days (valid range: 1-90)

# Remove existing PAT if it exists, then create a new one
try:
    session.sql(f"ALTER USER IF EXISTS {user} REMOVE PROGRAMMATIC ACCESS TOKEN {pat_name}").collect()
except:
    pass

# Generate new PAT
result = session.sql(f"""
    ALTER USER {user} 
    ADD PROGRAMMATIC ACCESS TOKEN {pat_name}
    DAYS_TO_EXPIRY = {days_to_expiry}
    COMMENT = 'PAT for Cortex Agents Lab'
""").collect()

# Extract the token from the result
if result:
    row = result[0].as_dict()
    pat_token = row.get('token_secret') or row.get('TOKEN_SECRET') or result[0][1]
else:
    pat_token = None

if pat_token:
    print(f"PAT generated successfully for user: {user}")
    print(f"Token expires in: {days_to_expiry} day(s)")
else:
    print("Failed to generate PAT")

### Set Up API Helper Function

Create a helper function that uses the PAT token to make authenticated REST API calls.

**Required Headers**:
- Authorization: Bearer <PAT_TOKEN>
- Content-Type: application/json
- X-Snowflake-Authorization-Token-Type: PROGRAMMATIC_ACCESS_TOKEN

In [ ]:
# Set up API helper function using PAT token authentication
import requests

# Get account URL from session
account_info = session.get_current_account().strip('"')
api_base = f"https://{account_info}.snowflakecomputing.com"

def snow_api_request(method, endpoint, body=None):
    headers = {
        'Authorization': f'Bearer {pat_token}',
        'Content-Type': 'application/json',
        'X-Snowflake-Authorization-Token-Type': 'PROGRAMMATIC_ACCESS_TOKEN'
    }
    
    url = f"{api_base}{endpoint}"
    
    if method.upper() == 'GET':
        response = requests.get(url, headers=headers)
    elif method.upper() == 'POST':
        response = requests.post(url, headers=headers, json=body)
    elif method.upper() == 'PUT':
        response = requests.put(url, headers=headers, json=body)
    elif method.upper() == 'DELETE':
        response = requests.delete(url, headers=headers)
    else:
        raise ValueError(f"Unsupported HTTP method: {method}")
    
    try:
        data = response.json() if response.content else None
    except requests.exceptions.JSONDecodeError:
        data = response.text if response.content else None
    
    return {
        'status': response.status_code,
        'data': data
    }

# Test the API connection
print(f"API Base URL: {api_base}")
test = snow_api_request("GET", f"/api/v2/databases/{user}_genai_db/schemas/resources/agents")

if test['status'] == 200:
    print(f"API connection successful! Found {len(test['data'])} existing agent(s)")
else:
    print(f"API connection failed. Status: {test['status']}")
    print(f"Response: {test['data']}")

## Creating a Cortex Agent using REST API

Now we'll create a Cortex Agent that uses our TravelBug search service. The agent will be able to answer questions about travel activities by searching through the PDF documents we processed earlier.

**REST API Endpoint**: `POST /api/v2/databases/{db}/schemas/{schema}/agents`

**Key Components**:
- **name**: Unique identifier for the agent
- **model**: The LLM to use (e.g., `claude-4-haiku`, `auto`)
- **tools**: List of tools the agent can use (Cortex Search services)
- **instructions**: System prompts that define agent behavior

### Define the Agent Configuration

The configuration specifies all the key aspects of the agent's behavior and capabilities:

| Property | Description |
|----------|-------------|
| **name** | Unique identifier for the agent |
| **model** | LLM to use (e.g., `claude-4-haiku`, `auto`) |
| **tools** | Array of tools the agent can invoke |
| **instructions.system** | Core system prompt defining persona |
| **instructions.response** | Guidelines for response format |
| **instructions.sample_questions** | Example Q&A pairs for guidance |

**Tool Configuration** (Cortex Search):
```json
"tools": [
  {
    "tool_spec": {
      "type": "cortex_search",
      "name": "tool_name"
    }
  }
],
"tool_resources": {
  "tool_name": {
    "name": "database.schema.SERVICE_NAME",
    "max_results": 5,
    "id_column": "..."
  }
}
```

In [ ]:
# Define the agent name and configuration
agent_name = f"{user.lower()}_travelbug_assistant"

agent_config = {
    "name": agent_name,
    "comment": "TravelBug AI Assistant for adventure activity questions",
    "instructions": {
        "response": "You are a helpful travel assistant for TravelBug adventures. Provide detailed, accurate information about activities, equipment, timing, and preparation. Always be friendly and encouraging about outdoor adventures.",
        "orchestration": "Use the search tool to find relevant information from TravelBug activity guides. If information is not available in the documents, clearly state that."
    },
    "models": {
        "orchestration": "auto"
    },
    "tools": [
        {
            "tool_spec": {
                "type": "cortex_search",
                "name": "travelbug_search"
            }
        }
    ],
    "tool_resources": {
        "travelbug_search": {
            "search_service": f"{user.lower()}_genai_db.resources.travelbug_search",
            "max_results": 5,
            "id_column": "doc_id"
        }
    }
}

print("Agent configuration:")
print(json.dumps(agent_config, indent=2))

### Create the agent using REST API.

This code creates a Cortex Agent via a REST API call.

Here are the key steps:

*   **Construct API Endpoint URL**:
The code builds the URL for the agents API endpoint using `api_base` and `user` variables.

*   **Define the API Call Function**:
A function named **create_agent_api_call** is defined to make the POST request. It takes the URL, agent config, and headers as arguments.

*   **Make the API Call**:
The function sends a POST request using `requests.post()` with the agent configuration as the JSON payload.

*   **Execute and Display Output**:
The function is called with the agent configuration, and the API response is printed in formatted JSON.

In [ ]:
# Create the agent via REST API
endpoint = f"/api/v2/databases/{user}_genai_db/schemas/resources/agents"

print(f"Creating agent: {agent_name}")
print(f"Configuration: {json.dumps(agent_config, indent=2)}")

response = snow_api_request("POST", endpoint, agent_config)

if response['status'] in [200, 201]:
    print(f"\n✓ Agent created successfully!")
    print(json.dumps(response['data'], indent=2))
else:
    print(f"\n✗ Agent creation failed. Status: {response['status']}")
    print(json.dumps(response['data'], indent=2))

### Verify the agent was created using SQL.

Let's confirm the agent exists in Snowflake by querying it directly with SQL.

In [ ]:
%%sql -r Verify_the_agent_was_created_using_SQL_sql
-- Verify the agent was created successfully
SHOW AGENTS IN SCHEMA {{user}}_genai_db.resources;

-- You can also describe the agent to see its configuration
-- DESCRIBE AGENT {{user}}_genai_db.resources.{{user}}_travelbug_assistant;

### Listing agents.

This code retrieves a list of agents from the specified database and schema via a REST API call.

Here are the key steps:

*   **URL Construction**: Constructs the full URL for the agents API endpoint using the database name (**{{user}}_genai_db**) and schema name (**resources**).

*   **Function Definition**: Defines a function called **list_agents_api_call** that takes a URL and headers dictionary as arguments and makes a GET request.

*   **API Call**: The function sends a GET request using `requests.get()` and returns the JSON response.

*   **Function Execution**: Calls the function and displays the list of agents returned from the API.

In [ ]:
# List all agents in the specified schema
endpoint = f"/api/v2/databases/{user}_genai_db/schemas/resources/agents"

print("Listing agents...")
response = snow_api_request("GET", endpoint)

if response['status'] == 200:
    print(f"\n✓ Found {len(response['data'])} agent(s):")
    print(json.dumps(response['data'], indent=2))
else:
    print(f"\n✗ Failed to list agents. Status: {response['status']}")
    print(json.dumps(response['data'], indent=2))

### Describing an agent.

This code retrieves detailed information about the **travelbug_assistant** agent via a REST API call.

Here are the key steps:

*   **Constructs the API Endpoint**  
   Builds the full REST API URL to fetch details about the agent, using the current user and database context.

*   **Defines the API Call Function**  
   A function **describe_agent_api_call** is created to make a GET request to retrieve agent details.

*   **Executes the Function**  
   The function is called with the URL and headers, and the response is stored.

*   **Displays the Agent Details**  
   The response is printed in a readable JSON format showing all attributes of the agent.

In [ ]:
# Get details of a specific agent
endpoint = f"/api/v2/databases/{user}_genai_db/schemas/resources/agents/{agent_name}"

print(f"Describing agent: {agent_name}")
response = snow_api_request("GET", endpoint)

if response['status'] == 200:
    print(f"\n✓ Agent details:")
    print(json.dumps(response['data'], indent=2))
else:
    print(f"\n✗ Failed to get agent details. Status: {response['status']}")
    print(json.dumps(response['data'], indent=2))

## Agent Interaction and Testing

Now let us test our agent by sending real queries via the REST API.

**Key Concepts**:
- **Thread**: A conversation session that maintains context across messages
- **Message**: A single user query or agent response
- **Parent Message ID**: Links messages in a conversation for context

**REST API Flow**:
1. Create a thread: POST /api/v2/cortex/threads
2. Send message: POST /api/v2/databases/{db}/schemas/{schema}/agents/{name}:run

### Create a Conversation Thread

A **thread** is a container for a conversation session. It:
- Maintains conversation history
- Enables multi-turn interactions
- Allows the agent to reference previous messages for context

**API Endpoint**: `POST /api/v2/cortex/threads`

**Request Body**:
```json
{
  "origin_application": "app_name"  // Optional: identifies the calling application
}
```

**Response**: Returns a `thread_id` used for all subsequent messages in this conversation.

In [ ]:
# Create a conversation thread for the agent
endpoint = "/api/v2/cortex/threads"
payload = {"origin_application": "cortex_agnt_lab"}

print("Creating conversation thread...")
response = snow_api_request("POST", endpoint, payload)

if response['status'] in [200, 201]:
    thread_id = response['data'].get("thread_id", response['data'].get("id", "unknown"))
    print(f"\n✓ Thread created: {thread_id}")
    print(json.dumps(response['data'], indent=2))
else:
    print(f"\n✗ Failed to create thread. Status: {response['status']}")
    print(json.dumps(response['data'], indent=2))
    thread_id = None

## Testing the TravelBug Assistant with sample questions.

The next cell sends questions to the TravelBug AI assistant using the REST API and displays the agent responses.

**Key Steps**:

1. **Define Test Questions**: A list of sample questions about activities like balloon rides and sunset tours

2. **Define Messaging Function**: The send_message_to_agent() function constructs the API endpoint and sends the message payload

3. **Execute Test Loop**: Each question is sent to the agent, and the response is displayed

In [ ]:
# Test questions for the TravelBug assistant
test_questions = [
    "What should I bring for the balloon ride?",
    "What time does the sunset tour start?",
]

def send_message_to_agent(thread_id, message, parent_message_id="0"):
    """Send a message to the agent and get a streaming SSE response."""
    endpoint = f"{api_base}/api/v2/databases/{user}_genai_db/schemas/resources/agents/{agent_name}:run"
    
    headers = {
        'Authorization': f'Bearer {pat_token}',
        'Content-Type': 'application/json',
        'Accept': 'text/event-stream',
        'X-Snowflake-Authorization-Token-Type': 'PROGRAMMATIC_ACCESS_TOKEN'
    }
    
    payload = {
        "thread_id": thread_id,
        "parent_message_id": parent_message_id,
        "messages": [
            {
                "role": "user",
                "content": [{"type": "text", "text": message}]
            }
        ]
    }
    
    response = requests.post(endpoint, headers=headers, json=payload, stream=True)
    
    if response.status_code != 200:
        return {"error": response.status_code, "message": response.text}
    
    final_response = None
    current_event = None
    
    for line in response.iter_lines():
        if line:
            line = line.decode('utf-8')
            if line.startswith('event:'):
                current_event = line[6:].strip()
            elif line.startswith('data:'):
                data_str = line[5:].strip()
                if data_str:
                    try:
                        data = json.loads(data_str)
                        if current_event == 'response' or 'content' in data:
                            final_response = data
                    except json.JSONDecodeError:
                        continue
    
    return final_response if final_response else {"error": "No response received"}

print("\n=== Agent Interaction Tests ===")
last_message_id = "0"
for i, question in enumerate(test_questions, 1):
    print(f"\n--- Test {i} ---")
    print(f"Q: {question}")
    response = send_message_to_agent(thread_id, question, last_message_id)
    
    if "error" in response:
        print(f"Error: {response}")
    else:
        agent_response = response.get("content", response.get("response", response))
        print(f"A: {json.dumps(agent_response, indent=2)}")
        last_message_id = response.get("message_id", last_message_id)

### Testing conversational context with follow-up questions.

This cell demonstrates multi-turn conversations with the TravelBug AI assistant to test its ability to maintain context.

**Key Steps**:

- **Define Follow-Up Questions**: A sequence of related questions about the balloon ride

- **Send Questions in Sequence**: Each question uses the same thread ID to maintain context

- **Display Responses**: Shows how the agent provides contextually aware answers

In [ ]:
# Test follow-up questions to demonstrate conversational context
follow_up_conversation = [
    "What should I bring for the balloon ride?",
    "How long does the balloon ride last?",
    "What if the weather is bad?",
    "Can I bring my camera?"
]

print("\n=== Follow-up Conversation Test ===")
parent_msg_id = "0"
for i, question in enumerate(follow_up_conversation, 1):
    print(f"\n--- Message {i} ---")
    print(f"User: {question}")
    response = send_message_to_agent(thread_id, question, parent_msg_id)
    
    if "error" in response:
        print(f"Error: {response}")
    else:
        agent_response = response.get("content", response.get("response", response))
        print(f"Assistant: {json.dumps(agent_response, indent=2)}")
        parent_msg_id = response.get("message_id", parent_msg_id)
        
        if i > 1:
            print(f"[Context: This is a follow-up in the same conversation thread]")

## Agent Updates and Advanced Configuration

Let's explore how to update agent configurations and add more advanced features.

### Updating the TravelBug Agent configuration.

This code updates the agent's configuration to refine its behavior, personality, and operational limits via a REST API call.

**Builds the API Endpoint**  
Constructs the REST API URL for updating the agent, using the current user, database, schema, and agent name.

**Defines the New Configuration**  
A configuration dictionary is created to specify how the agent should behave. It includes:

- **Instructions**:  
  - *Response*: Sets the tone and persona of the agent (enthusiastic, safety-focused, responsible tourism advocate).  
  - *Orchestration*: Guides how the agent uses tools and responds to multi-activity queries.  
  - *System*: Describes the agent's core expertise and priorities (e.g., safety, local regulations).  
  - *Sample Questions*: Provides examples to help the agent understand expected interactions.

- **Orchestration Limits**:  
  - *Budget*: Sets processing constraints, allowing up to 300 seconds and 7500 tokens per request.

**Defines the Update Function**  
A function **update_agent_api_call** is created to make a PUT request with the updated configuration.

**Executes the Update**  
The function is called with the new configuration and the API response is printed showing the update result.

### Register the Agent with Snowflake CoWork

Now that the agent is created and configured, let's register it with Snowflake CoWork so it can be used via the CoWork UI. This makes the agent accessible to users through the Snowflake CoWork interface.

If the agent is already registered, the exception handler ensures no error is raised.

In [ ]:
# Update agent configuration
endpoint = f"/api/v2/databases/{user}_genai_db/schemas/resources/agents/{agent_name}"

updated_config = {
    "instructions": {
        "response": "You are TravelBug's premium adventure assistant. Provide detailed, safety-focused information about activities. Always mention safety guidelines and encourage proper preparation. Be enthusiastic about outdoor adventures while emphasizing responsible tourism.",
        "orchestration": "Use the search tool to find comprehensive information. If multiple activities are mentioned, provide comparative information. Always include practical tips and local insights.",
        "system": "You are TravelBug's expert AI assistant. Specialize in adventure activities and prioritize customer safety and satisfaction. Include weather considerations and local regulations in your responses.",
        "sample_questions": [
            {
                "question": "What safety measures are in place for the balloon ride?",
                "answer": "I'll provide detailed information about our safety protocols, equipment checks, and weather monitoring procedures for balloon rides."
            },
            {
                "question": "Which activity is best for beginners?",
                "answer": "Let me compare our activities and recommend the most suitable options for first-time adventurers based on difficulty level and requirements."
            }
        ]
    },
    "orchestration": {
        "budget": {
            "seconds": 300,
            "tokens": 7500
        }
    }
}

print(f"Updating agent: {agent_name}")
print(f"New configuration: {json.dumps(updated_config, indent=2)}")

response = snow_api_request("PUT", endpoint, updated_config)

if response['status'] == 200:
    print(f"\n✓ Agent updated successfully!")
    print(json.dumps(response['data'], indent=2))
else:
    print(f"\n✗ Agent update failed. Status: {response['status']}")
    print(json.dumps(response['data'], indent=2))

In [ ]:
# Register agent with Snowflake Intelligence for CoWork UI access
from snowflake.snowpark.context import get_active_session

session = get_active_session()

try:
    session.sql(f"""
        ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT
        ADD AGENT {user}_genai_db.resources.{agent_name}
    """).collect()
    print(f"✓ Agent {agent_name} registered with Snowflake Intelligence successfully!")
except Exception as e:
    if "already" in str(e).lower() or "exists" in str(e).lower():
        print(f"Agent {agent_name} is already registered with Snowflake Intelligence. No action needed.")
    else:
        raise e

## 🎯 Challenge Questions

Test your understanding of the concepts covered in this lab.

In [ ]:
from snowflake.snowpark.context import get_active_session
from IPython.display import display, HTML

session = get_active_session()

quiz_data = [
    {"q": "What is the primary purpose of Cortex Agents?", "options": ["A) To replace all SQL queries with natural language", "B) To only perform sentiment analysis", "C) Cortex Agents orchestrate multiple tools like Cortex Search and Analyst to answer complex queries", "D) To generate training data for machine learning"], "hash": "04e946c8121b3c4f909a624635f0ef3c"},
    {"q": "What is the purpose of conversation threads in Cortex Agents?", "options": ["A) They store agent configuration settings", "B) Conversation threads maintain context across multiple interactions with an agent", "C) They define which tools the agent can use", "D) They are used only for logging purposes"], "hash": "6553e388c6e08f3d4e9826094f340332"},
    {"q": "How does Cortex Search enhance Cortex Agents?", "options": ["A) Cortex Search provides retrieval-augmented generation (RAG) capabilities for agents", "B) It replaces the need for semantic models", "C) It only searches for exact keyword matches", "D) It is required for all agent interactions"], "hash": "4ef3ae3b0861d5b4c7efc58288f1b679"},
    {"q": "What can you control through agent configuration?", "options": ["A) Only the LLM model to use", "B) Only the tools available to the agent", "C) Only the response format", "D) Agents can be configured with specific instructions to control their behavior and responses"], "hash": "28b8688fb39913f9104a20c2265299b6"},
    {"q": "What is the benefit of using REST API for Cortex Agents?", "options": ["A) It is the only way to create agents", "B) It provides faster response times", "C) REST API allows programmatic creation and management of Cortex Agents", "D) It eliminates the need for authentication"], "hash": "383efb511a5c65b923c6d68fcbe9544e"},
]

results_map = {}
for qi, item in enumerate(quiz_data):
    results_map[qi] = {}
    for opt in item["options"]:
        letter = opt[0]
        escaped_opt = opt.replace("'", "''")
        result = session.sql(f"CALL genai_db.resources.quiz_temp('{item['hash']}', '{escaped_opt}', 'False')").collect()
        feedback = result[0][0]
        is_correct = 'Correct' in feedback or '✅' in feedback
        results_map[qi][letter] = (feedback, is_correct)

html = """<style>
.cq { margin: 20px 0; padding: 16px; border: 1px solid #d0d0d0; border-radius: 10px; background: #fafafa; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; font-size: 14px; }
.cq h4 { font-family: inherit; }
.cq input[type="radio"] { display: none; }
.cq .lbl { display: block; padding: 8px 12px; border-radius: 6px; cursor: pointer; font-family: inherit; }
.cq .lbl:hover { background: #e8f0fe; }
.cq input[type="radio"]:checked + .lbl { border-color: #1a73e8; background: #e8f0fe; font-weight: 600; }
.cq .fb { display: none; padding: 6px 12px; margin-top: 2px; border-radius: 4px; font-weight: 600; font-family: inherit; }
.cq input[type="radio"]:checked + .lbl + .fb { display: block; }
.cq .fb.ok { background: #e6f4ea; color: #1e7e34; }
.cq .fb.no { background: #fce8e6; color: #c62828; }
</style>"""

for qi, item in enumerate(quiz_data):
    html += f'<div class="cq"><h4>Q{qi+1}: {item["q"]}</h4>'
    for opt in item["options"]:
        letter = opt[0]
        feedback, is_correct = results_map[qi][letter]
        css_class = 'ok' if is_correct else 'no'
        uid = f'cq{qi}_{letter}'
        html += f'<div class="opt"><input type="radio" name="cq{qi}" id="{uid}">'
        html += f'<label class="lbl" for="{uid}">{opt}</label>'
        html += f'<div class="fb {css_class}">{letter}) {feedback}</div></div>'
    html += '</div>'

display(HTML(html))

## Key Takeaways

❄️ **Cortex Agents REST API** provides programmatic control over agent lifecycle management, enabling automated deployment and configuration.

❄️ **Programmatic Access Tokens (PAT)** enable authenticated REST API calls from notebooks and applications.

❄️ **Agent Configuration** allows detailed customization of behavior, tools, budget and instructions to match specific business requirements.

❄️ **Tool Integration** enables agents to leverage Cortex Search services, providing contextual responses based on your organization's data.

❄️ **Conversational Context** is maintained across interactions within a thread, enabling natural, multi-turn conversations.

**Key REST API Endpoints**:
| Endpoint | Method | Description |
|----------|--------|-------------|
| `/api/v2/databases/{db}/schemas/{schema}/agents` | GET | List agents |
| `/api/v2/databases/{db}/schemas/{schema}/agents` | POST | Create agent |
| `/api/v2/databases/{db}/schemas/{schema}/agents/{name}` | GET | Describe agent |
| `/api/v2/databases/{db}/schemas/{schema}/agents/{name}` | PUT | Update agent |
| `/api/v2/cortex/threads` | POST | Create thread |
| `/api/v2/databases/{db}/schemas/{schema}/agents/{name}:run` | POST | Send message |